# Tutorial 03 — Cislunar manifold transport graph

Each L1/L2 libration orbit has stable+unstable manifold tubes — 4D surfaces in phase space
that carry trajectories ballistically toward (stable) or away from (unstable) the orbit.
Cut two tubes by a common Poincaré section; crossings are exact heteroclinic-class
connections with Δv equal to ||vx_A − vx_B|| (energy-consistent, machine-precise).

We demonstrate two cislunar patches:
- L1↔L2 halo at the x = 1−μ section (~112 m/s).
- NRHO↔L2 halo at the y = 0 section (~119 m/s) — the right section for NRHO since its
  Floquet multiplier is small enough that its tube barely reaches x = 1−μ in 12 nondim time.

_~30 seconds to run._

In [ ]:
import math, warnings
warnings.filterwarnings('ignore')
import numpy as np
import ariadne
from ariadne.orbits.halo import halo_family
from ariadne.connections.poincare_3d import tube_section_cut_3d, closest_approach_4d
from ariadne.dynamics.cr3bp import pseudo_potential

em = ariadne.system('EARTH_MOON')
MU, V_STAR = em.mu, em.V_star

In [ ]:
# Build the three cislunar orbits
l1 = halo_family(MU, point='L1', n=10, dz=4e-3, fam_n=40, lyap_amp0=2e-3, lyap_dx=4e-3)[5]
l2 = halo_family(MU, point='L2', n=10, dz=4e-3, fam_n=40, lyap_amp0=2e-3, lyap_dx=4e-3)[5]
nrho = ariadne.gateway_nrho()
print(f'L1 halo C={l1.jacobi:.4f}, L2 halo C={l2.jacobi:.4f}, NRHO C={nrho.jacobi:.4f}')

In [ ]:
# L1<->L2 patch on x = 1-mu (axis=0)
l1_u = tube_section_cut_3d(MU, l1, x_sec=1-MU, stable=False, n_seeds=200, t_max=12.0, axis=0)
l2_s = tube_section_cut_3d(MU, l2, x_sec=1-MU, stable=True,  n_seeds=200, t_max=12.0, axis=0)
r = closest_approach_4d(l1_u['yzvyvz'], l2_s['yzvyvz'])
pa, pb = r['point_a'], r['point_b']
y, z = 0.5*(pa[0]+pb[0]), 0.5*(pa[1]+pb[1])
vy_a, vz_a, vy_b, vz_b = float(pa[2]), float(pa[3]), float(pb[2]), float(pb[3])
om = pseudo_potential([1-MU, y, z, 0, 0, 0], MU)
vx_a = math.sqrt(2*om - l1.jacobi - vy_a**2 - vz_a**2)
vx_b = math.sqrt(2*om - l2.jacobi - vy_b**2 - vz_b**2)
dv12 = math.sqrt((vx_a-vx_b)**2 + (vy_a-vy_b)**2 + (vz_a-vz_b)**2) * V_STAR * 1000
print(f'L1 <-> L2 halo patch Δv = {dv12:.1f} m/s')

In [ ]:
# NRHO<->L2 patch on y = 0 (axis=1) -- the right section for NRHO
nu = tube_section_cut_3d(MU, nrho, x_sec=0.0, stable=False, branch=-1, n_seeds=160, t_max=12.0, axis=1)
ls = tube_section_cut_3d(MU, l2,   x_sec=0.0, stable=True,  branch=+1, n_seeds=160, t_max=12.0, axis=1)
r = closest_approach_4d(nu['yzvyvz'], ls['yzvyvz'])
pa, pb = r['point_a'], r['point_b']
x, z = 0.5*(pa[0]+pb[0]), 0.5*(pa[1]+pb[1])
vx_a, vz_a, vx_b, vz_b = float(pa[2]), float(pa[3]), float(pb[2]), float(pb[3])
om = pseudo_potential([x, 0, z, 0, 0, 0], MU)
vy_a = math.sqrt(2*om - nrho.jacobi - vx_a**2 - vz_a**2)
vy_b = math.sqrt(2*om - l2.jacobi   - vx_b**2 - vz_b**2)
dv_nrho = math.sqrt((vx_a-vx_b)**2 + (vy_a-vy_b)**2 + (vz_a-vz_b)**2) * V_STAR * 1000
print(f'NRHO <-> L2 halo patch Δv = {dv_nrho:.1f} m/s   (at x={x:.4f}, y=0, z={z:.4f})')
print('\nCislunar transport graph:')
print(f'  L1 halo --[{dv12:5.1f} m/s, x=1-μ]--> L2 halo')
print(f'  NRHO    --[{dv_nrho:5.1f} m/s, y=0  ]--> L2 halo')